# Reasoning prompts: when thinking helps, and what it costs
**A loan-eligibility checker, asked four ways — plain · step by step · step back · vote — and the price of each**

Session 14 was about *what* you say to a model. Today is about *whether to ask it to think*. You build a loan-eligibility checker for ten applicants, ask the same question four ways, and for each way read three numbers: how often is it right, what did it cost in tokens and seconds, and what would a thousand of these cost?

Before starting: `git pull`, `uv sync` (no new packages). **Budget:** this notebook spends somewhere between 40,000 and 80,000 of the 200,000 tokens a free Groq key gets per day (the meter under each section shows your real number), and the free plan allows about 8,000 tokens a minute, so some cells will pause and say they are waiting for the rate limit. That is normal. Every answer is remembered on disk, so **re-running a cell is free and gives the same numbers**; delete the memory only when you want a fresh draw. Numbers vary by model and by day: read your own tables.

## 1. Setup

In [2]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client, ollama_client                                   # Session 11
    from prompt_utils import PromptSpec, init_work_repo, save_to_library                      # Session 14
    from prompt_tasks import TASK1, TASK2, TASK3, classify_spec                               # Session 14
    from reasoning_tasks import (DEMO, LOAN_CASES, LOAN_POLICY, OTHER, STRETCH, classify_cot_spec, problem_card,
                                 score_classifier_lab)
    from reasoning_utils import (DiskCache, Lab, Meter, accuracy_markdown, case_table, check_s15_work, comparison_table,
                                 pass_rate, pick_hardest, pick_local_chat_model, pick_trouble, run_strategy,
                                 save_accuracy_table, strategy_summary, verdict, with_scratchpad)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14 and 15: run 'git pull' in the course folder, "
                      "then restart the kernel.") from e

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print("No key in .env: the sections that call the model will say 'skipped'. Everything else still runs.")

results = {}                       # (problem id, strategy) -> Attempt, so re-running a cell replaces rather than duplicates
hardest = LOAN_CASES[4]            # replaced in section 5 by the case that gave the most trouble

def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw)
    results[(problem.id, a.strategy)] = a
    return a

def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results]

model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\L460\.genai_course\s15_cache.json


## 2. The surprise: a one-line answer that costs hundreds of tokens
A warm-up from a shop: *a jacket costs Rs 2,000, take 20% off, then 10% off the reduced price, then a Rs 100 coupon.* We ask for **only the final answer, no working**.

**Predict first.** Before you run the cell, change `my_guess` to the number of *output tokens* you expect this one-line reply to cost.

In [4]:
print(problem_card(DEMO))
my_guess = 20          # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}")
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1340   | right answer: 1340   | correct: True
output tokens: 116   (you guessed 20)
   thinking you never saw: 102      the visible reply: 14
spent so far: 0 calls (2 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?*

## 3. The loan-eligibility checker: ten cases, two ways of asking
The policy is the same for every case. Ten applicants, each hiding a different trap: a boundary, a unit, a rule that fails quietly. The answer is always `approve` or `decline`, and Python worked out the right one independently, so every answer is marked right or wrong.

- **Plain**: "reply with only `FINAL: <answer>`, no working".
- **Step by step** (chain of thought), your *explainer*: "think step by step, show each calculation, then end with `FINAL: <answer>`".

**Predict first**: which cases do you expect the *plain* version to get wrong? Write their ids in `predict_wrong` before you run anything.

In [5]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []     # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

In [6]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL))
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print("\nyou predicted plain would get wrong:", predict_wrong, "\nit actually got wrong:             ", got_wrong)
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       approve ok        approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       approve ok        approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       approve ok        approve ok        
L07-existing-emis   decline       decline ok        decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       approve ok        approve ok        
L10-months          approve       approve ok        approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain        10/10         476       

*Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?*

## 4. Step back
**Step-back prompting** asks for the *principles* first, in a call that is not allowed to solve anything, and only then solves with those principles in front of it. Two calls instead of one. We try it on the three cases that gave the most trouble above (or, if nothing was wrong, the three that cost the most thinking).

In [7]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 3)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print("\nWhat the model wrote when it was only allowed to step back (first case):\n")
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L03-score-699', 'L05-age-at-end', 'L08-two-fail']

What the model wrote when it was only allowed to step back (first case):

- **EMI‑to‑income check**:  
  - Compute total monthly EMIs = existing EMI + new loan EMI.  
  - Compute 40 % of monthly income.  
  - Condition (a) holds if total EMIs ≤ 40 % of income.

- **Credit‑score threshold**:  
  - Condition (b) holds if credit score ≥ 700.

- **Age‑at‑term check**:  
  - Age at loan end = current age + loan term (in years).  
  - Condition (c) holds if this age ≤ 60.

- **All‑or‑nothing rule**:  
  - Approval requires *simultaneous* satisfaction of (a), (b), and (c).  
  - Failure of any single condition triggers a decline.

- **Common pitfalls**:  
  - Forgetting to add existing EMIs to the new EMI.  
  - Mis‑calculating 40 % of income (use decimal 0.40).  
  - Ignoring that the age check uses the *end* of the term, not the current age.

case                right answer  plain             cot               stepback     

*Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?*

## 5. Take a vote (self-consistency)
Ask the same step-by-step question several times at a temperature above zero, so the answers can differ, and take the **most common answer**. It is how you buy reliability with money: five times the calls for one answer. We spend it on one case only, the one that gave the most trouble.

In [8]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}")
    single = results.get((hardest.id, "cot"))
    if single:
        print(f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L08-two-fail - Two rules fail at once

votes: {'decline': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,926 tokens, 5s   vs one step-by-step call: 782 tokens, 3s

spent so far: 0 calls (33 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?*

## 6. The dial: reasoning effort
This model has a dial: `reasoning_effort` can be `"low"`, `"medium"` or `"high"`. The same plain question about the same case, only the dial changes. (Some models and providers do not have a dial; the cell says so instead of failing.)

In [9]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain")
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:
            print(f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}")
    if dial:
        print(comparison_table(dial, MODEL))
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L08-two-fail          plain/low    decline  ok         1     344       81    0.8     $0.05
L08-two-fail          plain/high   decline  ok         1     598      335    1.0     $0.12

spent so far: 0 calls (35 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


*Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?*

## 7. When thinking does not help
Back to Session 14's complaint classifier: eight messages, the bank's own convention (UPI problems are `account`, servicing requests are `other`). Three ways: the plain one-word version, the same with *thinking*, and the Session 14 version with **four worked examples**. Remember what the examples were for.

In [10]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(lab, classify_spec(0), mode="plain"),
        "zero-shot + thinking": score_classifier_lab(lab, classify_cot_spec(), mode="cot"),
        "four examples, one word": score_classifier_lab(lab, classify_spec(4), mode="plain"),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}")
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,549    5.6
zero-shot + thinking         6/8     1,892    3.7
four examples, one word      6/8     3,197    3.8

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 0 calls (59 answered from memory), 0 token

*Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?*

## 8. A different kind of model: your local Llama
Session 12's small local model does **not** think before it answers. Five of the loan cases, plain and step by step, now on your laptop. No tokens are billed, so this section does not touch your daily limit; it does take a few minutes on a laptop CPU, so start it and carry on reading. If Ollama is not running, the cell says so and moves on.

In [ ]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter())
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

*Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?*

## 9. Other domains (optional)
The same four ways of asking, on an insurance claim, a retail order and a factory defect puzzle, then (stretch) healthcare dosing with an invented formulary (not medical advice), telecom proration and an EMI formula. The EMI is a calculation a calculator does exactly: in Session 20 you give the model a tool for it. Leave `RUN_OTHER = False` to save tokens.

In [ ]:
RUN_OTHER = False
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL))
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

## 10. Reasoning behind a curtain
In a real product the customer must see the **answer**, not the working. So ask for the working inside `<scratchpad>` tags and the real reply inside `<final>` tags, and let your *code* strip the scratchpad. Two rules follow: the acceptance checks run on the `<final>` part only, and the scratchpad is **never shown or stored where a customer can read it**, because private working can mention things the reply must not (Task 2's confidential score is the example).

Below, the same reference prompts as Session 14, version 1 against version 2 with the curtain, run twice each at a temperature above zero. `BASE` is where your own Session 14 prompts would go: copy the parts from your `library.md` into a `PromptSpec` to improve *your* prompt.

In [ ]:
BASE = {TASK1.id: TASK1.reference, TASK2.id: TASK2.reference}      # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

In [ ]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS)
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS)
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call")
        print(f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call")
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print("   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer.")
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

*Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?*

## 11. Your work repository: the accuracy table and prompt library v2
Two things go into your **work** repository today.

1. **The accuracy table for the ten cases** (`experiments/s15_accuracy_table.md`), made from the results above. This is the session's checkpoint.
2. **Prompt library v2 for Tasks 1 and 2.** Same rule as Session 14: **never edit an old entry, add a new one.** Only save a v2 if it earned its place, and say why **in numbers** in `WHY` (for example `"4.0 vs 3.0 of 4 checks, 2.1x tokens, worth it for a credit decision"`).

The save cell stays off until you set `SAVE_V2 = True`, fill in both reasons, and have run sections 3 and 10.

In [ ]:
WORK_REPO = Path.home() / "Documents" / "genai-course-work"      # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print("Clone your work repository first (module02/work_repo_steps.md), then rerun this cell.")

In [ ]:
SAVE_V2 = False
WHY = {TASK1.id: "",      # YOUR TURN: one line, with numbers from section 10
       TASK2.id: ""}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(WORK_REPO, task.id, 2, with_scratchpad(BASE[task.id]), run_v2["final"], run_v2["results"],
                        purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. " + WHY[task.id])
    table = accuracy_markdown(results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL)
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print("not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)")

In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 15: accuracy table and prompt library v2"
git push
```
Then run the self-check below. It reads your repository.

In [ ]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

## 12. Your decision table
Fill this in from **your own numbers**, one word per cell: *use*, *skip*, or *only if* (and say what). It is the other deliverable of the session.

| Kind of question | plain | step by step | step back | vote x5 |
|---|---|---|---|---|
| Several rules chained, nothing hidden (L02, L07) | | | | |
| A rule that fails quietly (L05, L10) | | | | |
| Values exactly on a limit (L03, L04, L06) | | | | |
| A formula or a piece of theory to recall (R4) | | | | |
| Classification with your house rules (the complaints) | | | | |
| A calculation a calculator does exactly (S3, the EMI) | | | | |

## 13. Reflection
Answer questions 1-8 above, one or two lines each. Then one sentence: **when would you pay for thinking, and when would you refuse?**

1.
2.
3.
4.
5.
6.
7.
8.

Checkpoint: `uv run pytest tests/test_s15.py` is green, and the self-check above says `PASS` on all five lines. Push, and submit your repository link again.